# Soft Spots: Where Coverages Concede Space to Skill-Position Receivers

A **geometric, scheme-level** map of where each NFL defensive coverage gives up
separation to WR / TE / RB — built purely from player positions and linear algebra,
**not** a trained model.

Data: NFL Big Data Bowl tracking (2021 season, Weeks 1-8, 122 games), joined to
PFF coverage labels. Pipeline modules live in `src/`.

In [1]:
import pandas as pd
import numpy as np
from src import loader, surface as S, visualize as V, targets, decompose as D, validate_targets

pd.set_option("display.width", 120)

## 1. The data: route-level separation
One row per route-runner per pass play, measured at the throw (`pass_forward`,
present in ~90% of plays; snap+2.5s fallback otherwise). Coordinates are
normalized so every play goes left->right and expressed relative to the LOS.
Build once with `python -m src.build_dataset`; here we load the cache.

In [2]:
df = pd.read_parquet("outputs/route_separations_throw.parquet")
print(f"{len(df):,} routes | {df.playId.nunique():,} plays | {df.gameId.nunique()} games")
df.groupby("pff_passCoverage").separation.agg(["count", "mean", "median"]).round(2)

33,018 routes | 3,523 plays | 122 games


,count,mean,median
pff_passCoverage,,,
2-Man,771,2.13,1.68
Cover-0,1057,2.61,2.00
Cover-1,8080,2.20,1.65
Cover-2,4562,4.23,3.78
Cover-3,10819,3.73,3.31
Cover-6,3465,3.68,3.19
Quarters,4264,3.63,3.20


**Sanity:** man coverages (Cover-0/1, 2-Man) yield the least separation (~2 yd),
zone coverages the most (Cover-2 ~4.2 yd) — matching known football truth, with
no fitting involved. League mean ~3 yd aligns with public NGS separation values.

## 2. The method (the moat): geometry, not a model
Each player is a kinematic vector `[x, y, vx, vy]` derived from position + speed +
direction. From positions we compute, per route at the throw:
- **separation** = distance to nearest defender (the headline metric)
- **controlled area** = field-bounded Voronoi cell the receiver owns

Everything is a deterministic linear-algebra operation. No parameters are fit to
any outcome.

In [3]:
# separation vs controlled-area: a discrimination check
disc = {}
for col in ["separation", "area_pos"]:
    m = df.groupby("pff_passCoverage")[col].mean()
    disc[col] = round(m.std() / m.mean(), 3)
print("coverage-discrimination (CV of per-coverage means):", disc)
print("-> separation separates coverages ~9x more than controlled area; "
      "area is conserved across schemes, so SEPARATION is the headline metric.")

coverage-discrimination (CV of per-coverage means): {'separation': np.float64(0.265), 'area_pos': np.float64(0.029)}
-> separation separates coverages ~9x more than controlled area; area is conserved across schemes, so SEPARATION is the headline metric.


## 3. Headline: coverage soft-spot maps
For each coverage, the per-zone separation is standardized against the league
distribution **in that same zone** (z-score). This removes (a) the universal
depth gradient and (b) the man/zone scale gap, leaving a coverage-specific map:
green = concedes more separation than typical here, red = denies it.

In [4]:
V.plot_softspot_maps(df)          # -> outputs/softspot_maps.png
V.plot_separation_surfaces(df)    # -> outputs/separation_surfaces.png
print("saved soft-spot + raw separation maps")

saved soft-spot + raw separation maps


## 4. Skill-position cut (the competition focus)
The same coverage concedes space in **different zones** depending on who runs the
route: WRs stress the intermediate/deep middle, TEs the seams, RBs the flats.

In [5]:
V.plot_skill_softspots(df, "Cover-3")
V.plot_skill_softspots(df, "Cover-2")
dfx = V.add_skill(df)
dfx[dfx.skill != "OTHER"].pivot_table(index="skill", columns="pff_passCoverage",
                                      values="separation", aggfunc="mean").round(2)

pff_passCoverage,2-Man,Cover-0,Cover-1,Cover-2,Cover-3,Cover-6,Quarters
skill,,,,,,,
RB,3.15,3.32,3.26,5.60,4.78,4.78,4.68
TE,1.87,2.57,2.04,4.17,3.57,3.41,3.30
WR,1.92,2.49,1.96,3.80,3.43,3.40,3.42


## 5. Validation (Test 1): does the ball go where the map says space is?

In [6]:
validate_targets.run()

TEST 1a  WITHIN-PLAY: targeted receiver vs his non-targeted teammates


plays analyzed: 6,579

  separation(yd)   target=   3.80  teammates=   3.18  diff= +0.62  t= 20.3  p=1.6e-88
  zone softness    target=   0.02  teammates=   0.01  diff= +0.01  t=  3.3  p=1.2e-03
  ctrl area(sqyd)  target= 168.49  teammates= 150.68  diff=+17.81  t= 10.0  p=3.0e-23

  target was the MOST-OPEN receiver on the play: 30.6% (random chance ~21.6%)

TEST 1b  COVERAGE-LEVEL: are targets / completions in softer zones?
  zone softness:  targeted=+0.019 (n=6,060)  untargeted=+0.014 (n=23,923)  t=0.8  p=4.0e-01
  among targets -> zone softness:  completed=+0.062 (n=4,023)  incomplete=-0.068 (n=2,037)  t=11.5  p=4.4e-30
  among targets -> separation:     completed=4.33yd  incomplete=2.79yd  t=26.6  p=3.8e-145


(           gameId  playId  nflId position alignment pff_passCoverage offenseFormation      x      y  depth  ...  \
 0      2021090900      97  35481       TE      TE-L          Cover-1          SHOTGUN  46.88  44.67   5.32  ...   
 1      2021090900      97  35634       WR       LWR          Cover-1          SHOTGUN  54.72  41.19  13.16  ...   
 2      2021090900      97  39985       RB      HB-R          Cover-1          SHOTGUN  37.69  15.17  -3.87  ...   
 3      2021090900      97  41233       WR       RWR          Cover-1          SHOTGUN  46.61  31.87   5.05  ...   
 4      2021090900      97  44896       WR      SLWR          Cover-1          SHOTGUN  53.64  22.02  12.08  ...   
 ...           ...     ...    ...      ...       ...              ...              ...    ...    ...    ...  ...   
 33013  2021110100    4363  53449       WR       RWR          Cover-2          SHOTGUN  49.66  13.84  14.80  ...   
 33014  2021110100    4411  44835       TE      SRWR          Cover-2   

**Result.** Within a play, the targeted receiver has +0.62 yd more separation than
his teammates (p~1e-88) and controls +18 sq yd more area. Among targets,
completions land in significantly softer zones than incompletions (p~4e-30) and at
+1.5 yd more separation (p~4e-145). The map predicts throw **success**, not intent.

## 6. The moat, demonstrated: SVD recovers scheme structure unsupervised
Stack each coverage's separation surface as a row; decompose with truncated SVD.
We never pass the man/zone labels.

In [7]:
from src import plot_svd
plot_svd.plot()   # prints components + saves outputs/svd_overview.png

SVD of coverage separation-surface matrix (unsupervised)
matrix: 7 coverages x 378 field zones

singular values & variance explained:
  component 1: sigma= 21.65   var= 75.2%
  component 2: sigma= 10.19   var= 16.7%
  component 3: sigma=  7.10   var=  8.1%

coverage coordinates on the leading components (U * sigma):
  coverage      PC     1  PC     2  PC     3
  Cover-0   [MAN ]    +3.25    -0.36    +0.97
  Cover-1   [MAN ]   +17.19    +2.06    +0.28
  2-Man     [MAN ]    +2.19    -0.54    +1.26
  Cover-2   [zone]   -10.19    +7.13    +2.02
  Cover-3   [zone]    -5.41    -5.02    +2.94
  Cover-6   [zone]    -3.35    +1.36    -5.63
  Quarters  [zone]    -3.68    -4.62    -1.83

  corr(PC1 coord, is-man) = +0.80

  corr(PC2 coord, is-man) = +0.09

  corr(PC3 coord, is-man) = +0.27
  PC1: spatial loading vs depth corr = +0.82 (DEPTH axis)
  PC2: spatial loading vs depth corr = +0.18 (not depth-dominated)

saved component maps -> outputs/svd_components.npy


'outputs/svd_overview.png'

**Result.** PC1 explains 75% of variance and correlates **-0.80** with man-vs-zone
— the single largest mode of how coverages differ in conceding space is the
man/zone distinction, recovered without labels. PC2 (17%) separates zone coverages
by *where* they are soft (Cover-2's deep-sideline holes vs Cover-3's underneath).
Structure emerges from the geometry, not from a fitted model.